# Notebook 02: Feature Stability Under PGD Attack (Experiment 1)

For each ε ∈ [0.01, 0.05, 0.1, 0.5]:
- Run 20-step PGD in embedding space to **maximise SAE disruption**
- Compare against random Gaussian baseline at the same ε
- Metrics: cosine similarity, Jaccard index, rank correlation, feature flip count, KL divergence

Results → `results/exp1_feature_stability.json`

In [ ]:
!pip install transformer-lens==1.19.0 sae-lens==4.3.1 transformers==4.44.2 \
    datasets==2.20.0 tqdm==4.66.4 scipy==1.13.1 -q

In [ ]:
import os, sys, json, random, time
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/rajo69/Adversarial-Robustness-of-SAE-Interpretations"
REPO_DIR = "/content/SAE_Adversarial_Project"
if not os.path.exists(REPO_DIR):
    os.system(f"git clone {GITHUB_REPO_URL} {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")
sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f"CWD: {os.getcwd()}")

In [ ]:
import numpy as np
import torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
RESULTS_DIR = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR  = Path("figures");  FIGURES_DIR.mkdir(exist_ok=True)
print(f"Device: {DEVICE}")

In [ ]:
from transformer_lens import HookedTransformer
from sae_lens import SAE

try:
    model = HookedTransformer.from_pretrained("gemma-2-2b", device=DEVICE)
    MODEL_NAME = "gemma-2-2b"; SAE_RELEASE = "gemma-scope-2b-pt-res"
    SAE_ID_TMPL = "layer_{layer}/width_16k/average_l0_82"
    HOOK_TMPL = "blocks.{layer}.hook_resid_post"; TARGET_LAYER = 12
except Exception as e:
    print(f"Gemma fallback ({e})")
    model = HookedTransformer.from_pretrained("gpt2", device=DEVICE)
    MODEL_NAME = "gpt2"; SAE_RELEASE = "gpt2-small-res-jb"
    SAE_ID_TMPL = "blocks.{layer}.hook_resid_pre"
    HOOK_TMPL = "blocks.{layer}.hook_resid_pre"; TARGET_LAYER = 8

model.eval()
sae, _, _ = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID_TMPL.format(layer=TARGET_LAYER))
sae = sae.to(DEVICE); sae.eval()
print(f"Model: {MODEL_NAME} | Layer: {TARGET_LAYER}")
if DEVICE == "cuda": print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
from src.data import load_eval_set, tokens_to_tensor

eval_set = load_eval_set()
# Reproducible 100-prompt subset
torch.manual_seed(SEED)
idx = sorted(torch.randperm(len(eval_set))[:100].tolist())
exp_set = [eval_set[i] for i in idx]
print(f"Experiment subset: {len(exp_set)} prompts")

In [ ]:
EPSILON_VALUES  = [0.01, 0.05, 0.1, 0.5]
PGD_STEPS       = 20
N_RANDOM        = 10   # random baseline samples per prompt

print(f"Epsilons: {EPSILON_VALUES}  |  PGD steps: {PGD_STEPS}  |  Random samples: {N_RANDOM}")
print(f"Total PGD runs: {len(EPSILON_VALUES) * len(exp_set)}")

In [ ]:
from src.attacks import pgd_attack_sae, random_perturbation_baseline
from src.metrics import compute_all_metrics

results_per_eps = {}

for eps in EPSILON_VALUES:
    print(f"\n{'='*55}\n  ε = {eps}\n{'='*55}")
    adv_list, rnd_list = [], []
    hn = HOOK_TMPL.format(layer=TARGET_LAYER)

    for item in tqdm(exp_set, desc=f"ε={eps}"):
        tok = tokens_to_tensor(item["tokens"], device=DEVICE)

        # Clean activations
        with torch.no_grad():
            cl_logits, cl_cache = model.run_with_cache(tok, names_filter=hn)
            cl_acts = sae.encode(cl_cache[hn])

        # PGD attack
        pert_emb = pgd_attack_sae(
            model=model, sae=sae, clean_tokens=tok,
            target_layer=TARGET_LAYER, epsilon=eps,
            steps=PGD_STEPS, device=DEVICE,
        )

        # Perturbed forward pass
        with torch.no_grad():
            def _h(v, h): return pert_emb
            pt_logits, pt_cache = model.run_with_cache(
                tok, names_filter=hn, fwd_hooks=[("hook_embed", _h)])
            pt_acts = sae.encode(pt_cache[hn])

        adv_list.append(compute_all_metrics(cl_acts, pt_acts, cl_logits, pt_logits))

        # Random baseline
        rnd_list.append(random_perturbation_baseline(
            model=model, sae=sae, clean_tokens=tok,
            target_layer=TARGET_LAYER, epsilon=eps,
            n_samples=N_RANDOM, seed=SEED, device=DEVICE,
        ))

        if DEVICE == "cuda": torch.cuda.empty_cache()

    results_per_eps[str(eps)] = {"adversarial": adv_list, "random": rnd_list}
    cos_m = np.mean([m["cosine_similarity"] for m in adv_list])
    kl_m  = np.mean([m["kl_divergence"]     for m in adv_list])
    print(f"  adv cosine={cos_m:.4f}  output KL={kl_m:.4f}")

In [ ]:
summary = {}
for eps_s, data in results_per_eps.items():
    adv = data["adversarial"]; rnd = data["random"]
    ac = [m["cosine_similarity"] for m in adv]
    rc = [m["mean_cosine"]       for m in rnd]
    aj = [m["jaccard_index"]     for m in adv]
    af = [m["feature_flip_count"]for m in adv]
    ak = [m["kl_divergence"]     for m in adv]
    # adversarial advantage: ratio of disruption vs random
    adv_adv = [(1-a)/(1-r+1e-10) for a, r in zip(ac, rc)]
    summary[eps_s] = {
        "adv_cosine_mean":  float(np.mean(ac)),  "adv_cosine_std":  float(np.std(ac)),
        "rnd_cosine_mean":  float(np.mean(rc)),  "rnd_cosine_std":  float(np.std(rc)),
        "adv_jaccard_mean": float(np.mean(aj)),  "adv_jaccard_std": float(np.std(aj)),
        "adv_flip_mean":    float(np.mean(af)),  "adv_flip_std":    float(np.std(af)),
        "adv_kl_mean":      float(np.mean(ak)),  "adv_kl_std":      float(np.std(ak)),
        "adv_advantage_mean": float(np.mean(adv_adv)),
        "adv_advantage_std":  float(np.std(adv_adv)),
    }
    s = summary[eps_s]
    print(f"ε={eps_s}: adv_cos={s['adv_cosine_mean']:.4f} rnd_cos={s['rnd_cosine_mean']:.4f} "
          f"advantage={s['adv_advantage_mean']:.2f}×")

In [ ]:
output = {
    "experiment": "feature_stability", "model": MODEL_NAME,
    "sae_release": SAE_RELEASE, "target_layer": TARGET_LAYER,
    "n_prompts": len(exp_set), "pgd_steps": PGD_STEPS,
    "n_random_samples": N_RANDOM, "seed": SEED,
    "epsilon_values": EPSILON_VALUES, "summary": summary,
    "per_prompt": {
        eps_s: {"adversarial": data["adversarial"],
                "random": data["random"]}
        for eps_s, data in results_per_eps.items()
    },
}
p = RESULTS_DIR / "exp1_feature_stability.json"
p.write_text(json.dumps(output, indent=2))
print(f"Saved: {p}")

In [ ]:
from src.plot_config import apply_style, save_fig, COLORS, EPSILON_COLORS, epsilon_label

apply_style()
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
fig.suptitle(f"Exp 1: Feature Stability Under PGD\n{MODEL_NAME}, Layer {TARGET_LAYER}", fontsize=13)

eps_f = [float(e) for e in summary]
ac_m  = [summary[s]["adv_cosine_mean"]  for s in summary]
ac_s  = [summary[s]["adv_cosine_std"]   for s in summary]
rc_m  = [summary[s]["rnd_cosine_mean"]  for s in summary]
rc_s  = [summary[s]["rnd_cosine_std"]   for s in summary]
aj_m  = [summary[s]["adv_jaccard_mean"] for s in summary]
aj_s  = [summary[s]["adv_jaccard_std"]  for s in summary]
aa_m  = [summary[s]["adv_advantage_mean"] for s in summary]

# 1) Cosine similarity vs epsilon
ax = axes[0,0]
ax.errorbar(eps_f, ac_m, yerr=ac_s, fmt='o-', color=COLORS["adversarial"],
            label="PGD", capsize=4)
ax.errorbar(eps_f, rc_m, yerr=rc_s, fmt='s--', color=COLORS["random"],
            label="Random", capsize=4)
ax.set_xscale("log"); ax.set_ylim([-0.1, 1.05])
ax.set_xlabel("ε"); ax.set_ylabel("Cosine Similarity")
ax.set_title("SAE Cosine Similarity vs ε"); ax.legend()

# 2) Jaccard index vs epsilon
ax = axes[0,1]
ax.errorbar(eps_f, aj_m, yerr=aj_s, fmt='o-', color=COLORS["adversarial"], capsize=4)
ax.set_xscale("log"); ax.set_ylim([-0.05, 1.05])
ax.set_xlabel("ε"); ax.set_ylabel("Jaccard Index")
ax.set_title("Active Feature Overlap vs ε")

# 3) MONEY PLOT: SAE disruption vs output KL
ax = axes[1,0]
for eps_s, data in results_per_eps.items():
    ef = float(eps_s)
    x  = [m["kl_divergence"]  for m in data["adversarial"]]
    y  = [m["sae_disruption"] for m in data["adversarial"]]
    ax.scatter(x, y, alpha=0.5, s=18,
               color=EPSILON_COLORS.get(ef, COLORS["neutral"]),
               label=epsilon_label(ef))
ax.set_xlabel("Output KL Divergence (↓ = output preserved)")
ax.set_ylabel("SAE Disruption (1 − cosine sim, ↑ = more disrupted)")
ax.set_title("MONEY PLOT: SAE Disruption vs Output Change\n"
             "top-left = SAE fooled while output intact")
ax.legend(loc="upper left", fontsize=8)

# 4) Adversarial advantage
ax = axes[1,1]
ax.bar(range(len(eps_f)), aa_m,
       color=[EPSILON_COLORS.get(e, COLORS["neutral"]) for e in eps_f])
ax.axhline(1.0, color="black", ls="--", lw=1.2, label="Random baseline (1×)")
ax.set_xticks(range(len(eps_f)))
ax.set_xticklabels([f"ε={e}" for e in eps_f])
ax.set_ylabel("Adversarial Advantage")
ax.set_title("PGD vs Random: How Exploitable Is the Vulnerability?"); ax.legend()

plt.tight_layout()
fp = FIGURES_DIR / "exp1_feature_stability.png"
save_fig(fig, str(fp)); plt.show()
print(f"Saved: {fp}")

## ✓ Experiment 1 complete

**Record in `progress.md`:** cosine similarity at each ε (adv vs random), adversarial advantage,
whether the money plot shows a top-left cluster.

Next: `03_output_preserving.ipynb` (novel contribution).